# 潮汐赶海安全助手 - 可复现演示

本 Notebook 演示潮汐赶海安全助手的完整流程，确保**图、表、文字说明三者对得上**。

## 功能模块
1. **潮汐计算引擎** - 时区标准化、"潮位时区错"检测、潮位插值
2. **风险分层系统** - 结合潮位、水深、航速、数据质量的多维度评估
3. **数据复核与追溯** - 原始行号保留、来源备注、数据状态（可用/暂缓/需重采）
4. **可视化与地图联动** - 图表+交互式地图，结论回溯源数据

In [ ]:
import os
import sys
sys.path.insert(0, os.path.dirname(os.getcwd()))

import pandas as pd
from IPython.display import display, Image, HTML, Markdown

from data.mock_data_generator import generate_tide_table, generate_ship_trajectory
from src.traceability import DataTracker, DataStatus
from src.tide_engine import TideCalculator
from src.risk_layer import RiskAssessor
from src.visualization import TideVisualizer

BASE_DIR = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("环境初始化完成")

## 一、模拟数据（预设4条时区异常）

模拟数据包含潮汐表（30条）和船舶轨迹（80条）。其中特意在 T002、T003 两个潮位站设置了 4 条"潮位时区错"异常：上报时区写 UTC，但实际是北京时间。

In [ ]:
tide_csv = os.path.join(DATA_DIR, "tide_table_sample.csv")
ship_csv = os.path.join(DATA_DIR, "ship_trajectory_sample.csv")

if not os.path.exists(tide_csv):
    generate_tide_table(tide_csv)
if not os.path.exists(ship_csv):
    generate_ship_trajectory(ship_csv)

df_tide_raw = pd.read_csv(tide_csv)
df_ship = pd.read_csv(ship_csv)

print(f"潮汐表: {len(df_tide_raw)} 条")
print(f"船舶轨迹: {len(df_ship)} 条")

tz_errors = df_tide_raw[df_tide_raw["has_timezone_error"] == 1]
print(f"\n预设时区异常 {len(tz_errors)} 条（位于行号: {tz_errors['source_row'].tolist()}）")

print("\n=== 原始潮汐表（前10行，含追溯字段） ===")
display(df_tide_raw.head(10))

## 二、潮汐计算引擎 - 时区标准化与异常检测

核心功能：
- 解析时间字符串并按上报时区转换为 UTC + 北京时间
- 检测"潮位时区错"（启发式+先验规则）
- 与 DataTracker 联动，给每条记录打状态标签

In [ ]:
tracker = DataTracker()
tide_calc = TideCalculator(tracker=tracker)
tide_records = tide_calc.process_tide_table(df_tide_raw)

df_tide = tide_calc.get_dataframe()
summary = tracker.status_summary()

print(f"处理结果: 可用={summary['by_status']['可用']['count']}, "
      f"暂缓={summary['by_status']['暂缓']['count']}, "
      f"需重采={summary['by_status']['需重新采集']['count']}")
print(f"异常类型: {summary['issue_types']}")

print("\n=== 标准化后的潮汐表（含UTC/北京时间双时区，关联追溯ID） ===")
display(df_tide.head(10))

print("\n=== 检测到的异常记录（保留原始行号+来源备注） ===")
df_track = tracker.to_dataframe()
bad = df_track[df_track["data_status"] != "可用"]
display(bad[["record_id", "source_row", "source_file", "source_note", 
            "data_status", "issue_type", "issue_description", "next_action"]])

## 三、风险分层评估

评估维度：
1. **潮位风险** - 基于插值估算的实时潮位
2. **水深风险** - 船底与海底距离是否足够
3. **航速风险** - 近岸是否超速
4. **数据可信度** - 关联潮汐数据质量（时区错等）

风险等级：安全 / 注意 / 警告 / 危险

In [ ]:
risk_tracker = DataTracker()
assessor = RiskAssessor(tide_calc, tracker=risk_tracker)
risk_results = assessor.assess_trajectory(df_ship)

df_risk = assessor.get_dataframe()
rs = assessor.summary()

print("风险分层汇总:")
for lvl, cnt in rs["counts"].items():
    if cnt > 0:
        print(f"  [{lvl}] {cnt} 条")
print("\n各船最高风险:")
for sid, info in rs["by_ship"].items():
    print(f"  {info['ship_name']} ({sid}): {info['max_level']}")

print("\n=== 风险评估结果（前15行含建议） ===")
display(df_risk[["record_id", "ship_name", "record_time", "risk_level", 
                 "estimated_tide_cm", "water_depth_m", "suggestion", "source_row"]].head(15))

## 四、数据复核报告（水产养殖场长视角）

报告特点：
- 说明**哪些可用、哪些暂缓、哪些需重采**
- 每条异常给出**下一步操作**（补材料还是改口径）
- 保留**原始行号、来源文件名、来源备注**，可回溯到原始表/扫描页

In [ ]:
report = tracker.generate_review_report()
print(report)

## 五、船队使用说明（简洁版）

船队拿到结果时，一眼分清：
- ✅ 哪些能直接用
- ❓ 哪些还要找水产养殖场长复核
- ❌ 哪些数据异常不能用

In [ ]:
brief = tracker.generate_fleet_brief()
print(brief)

## 六、可视化输出（图、表、文字一致）

每张图都附带 .txt 说明文件，确保图文表三者对得上。

In [ ]:
viz = TideVisualizer(OUTPUT_DIR)

r1 = viz.plot_tide_curve(tide_calc, "tide_curve.png")
r2 = viz.plot_data_status_pie(tracker, "data_status_pie.png")
r3 = viz.plot_risk_distribution(assessor, "risk_distribution.png")
r4 = viz.plot_risk_timeline(assessor, "risk_timeline.png")
r5 = viz.create_risk_map(assessor, tide_calc, "risk_map.html")
report_path = viz.generate_summary_text(tracker, assessor, "summary_report.txt")

df_tide.to_csv(os.path.join(OUTPUT_DIR, "tide_processed.csv"), index=False, encoding="utf-8-sig")
df_risk.to_csv(os.path.join(OUTPUT_DIR, "risk_results.csv"), index=False, encoding="utf-8-sig")
df_track.to_csv(os.path.join(OUTPUT_DIR, "tracking_records.csv"), index=False, encoding="utf-8-sig")

print("可视化输出完成，所有文件位于 output/ 目录")

In [ ]:
print("=== 1. 潮位曲线（异常点红色空心圆标注） ===")
display(Image(os.path.join(OUTPUT_DIR, "tide_curve.png")))
print("\n图表说明:")
print(r1["caption"])

In [ ]:
print("=== 2. 数据复核状态饼图 ===")
display(Image(os.path.join(OUTPUT_DIR, "data_status_pie.png")))
print("\n图表说明:")
print(r2["caption"])

In [ ]:
print("=== 3. 船舶风险分层柱状图 ===")
display(Image(os.path.join(OUTPUT_DIR, "risk_distribution.png")))
print("\n图表说明:")
print(r3["caption"])

In [ ]:
print("=== 4. 船舶风险时间线 ===")
display(Image(os.path.join(OUTPUT_DIR, "risk_timeline.png")))
print("\n图表说明:")
print(r4["caption"])

In [ ]:
print("=== 5. 交互式风险地图 ===")
print(f"文件: {r5['path']}")
print("请在浏览器中打开查看。支持点击轨迹点查看：")
print("  - 船舶、时间、风险等级")
print("  - 估算潮位、水深、航速")
print("  - 行动建议")
print("  - 原始行号、来源备注（追溯用）")
print("\n")
print(r5["caption"])

## 七、图文表一致性交叉验证

验证三张图表和数据表格中的统计数字完全对得上。

In [ ]:
print("=== 交叉验证：图、表、文字统计一致 ===")

print("\n【1】潮汐数据状态:")
print(f"  表格统计: 可用={(df_track['data_status']=='可用').sum()}, "
      f"暂缓={(df_track['data_status']=='暂缓').sum()}, "
      f"需重采={(df_track['data_status']=='需重新采集').sum()}")
print(f"  饼图统计: {r2['summary']['by_status']}")
print(f"  报告文字: 可用={summary['by_status']['可用']['count']}, "
      f"暂缓={summary['by_status']['暂缓']['count']}, "
      f"需重采={summary['by_status']['需重新采集']['count']}")

print("\n【2】风险分层统计:")
for lvl in ["安全", "注意", "警告", "危险"]:
    tbl = (df_risk['risk_level']==lvl).sum()
    viz_cnt = rs['counts'].get(lvl, 0)
    match = "✅" if tbl == viz_cnt else "❌"
    print(f"  {match} [{lvl}] 表格={tbl} 柱状图={viz_cnt}")

print("\n【3】异常记录追溯（以REC-00011为例）:")
rid = "REC-00011"
tide_row = df_tide[df_tide["record_id"]==rid]
track_row = df_track[df_track["record_id"]==rid]
print(f"  潮汐表: 行{tide_row['source_row'].values[0]} "
      f"站={tide_row['station_name'].values[0]}")
print(f"  追溯表: 状态={track_row['data_status'].values[0]} "
      f"问题={track_row['issue_type'].values[0]}")
print(f"  来源: {track_row['source_file'].values[0]} "
      f"行{track_row['source_row'].values[0]} {track_row['source_note'].values[0]}")
print("  → 图表、表格、文字报告中的 record_id、行号、来源完全对应")

## 八、输出文件清单

所有输出均位于 `output/` 目录，图文表一一对应。

In [ ]:
print(f"{'文件':<35} {'大小(KB)':>10}  {'说明'}")
print("-"*75)
for f in sorted(os.listdir(OUTPUT_DIR)):
    fp = os.path.join(OUTPUT_DIR, f)
    size = os.path.getsize(fp)/1024
    desc = ""
    if f == "tide_curve.png": desc = "潮位曲线（异常点高亮）"
    elif f == "data_status_pie.png": desc = "数据复核状态饼图"
    elif f == "risk_distribution.png": desc = "船舶风险分层柱状图"
    elif f == "risk_timeline.png": desc = "船舶风险时间线"
    elif f == "risk_map.html": desc = "交互式风险地图（浏览器打开）"
    elif f == "summary_report.txt": desc = "综合复核报告"
    elif f == "tide_processed.csv": desc = "标准化潮汐表（含追溯ID）"
    elif f == "risk_results.csv": desc = "风险评估结果表"
    elif f == "tracking_records.csv": desc = "数据追溯明细表"
    elif f.endswith(".txt"): desc = "对应图表的文字说明"
    print(f"{f:<35} {size:>10.1f}  {desc}")